<a href="https://colab.research.google.com/github/meechkb/retail-sales-forecasting-pipeline/blob/main/Retail_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, GRU, Input, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.'
file_path = os.path.join(BASE_DIR, 'data', 'raw', '211FinalProject.xlsx')
if not os.path.exists(file_path):
    raise FileNotFoundError("Dataset not found. Please place it in data/raw/")
sns.set(style="whitegrid")

# Load each sheet
df_2009 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
df_2010 = pd.read_excel(file_path, sheet_name='Year 2010-2011')

# Combine sheets
df = pd.concat([df_2009, df_2010], ignore_index=True)

# Step 1: Keep only the necessary columns from the original dataset
df_clean = df[["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID"]].copy()

# Step 2: Remove rows with missing values
df_clean.dropna(subset=["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID"], inplace=True)

# Step 3: Remove cancellations
df = df[~df["Invoice"].astype(str).str.startswith("C")].copy()

# Step 4: Keep only positive quantity and price
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

# Step 5: Create Sales column
df["Sales"] = df["Quantity"] * df["Price"]

# Step 6: Convert InvoiceDate to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Step 7: Sort by date
df.sort_values("InvoiceDate", inplace=True)

# Step 8-12: Aggregate daily sales, fill missing days with 0
daily_sales = df.groupby(df["InvoiceDate"].dt.date)["Sales"].sum().reset_index()
daily_sales.columns = ["Date", "Sales"]
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])
daily_sales.set_index("Date", inplace=True)
daily_sales = daily_sales.asfreq("D")
daily_sales["Sales"] = daily_sales["Sales"].fillna(0)

# Step 13: Add time-based features
daily_sales["dayofweek"] = daily_sales.index.dayofweek
daily_sales["month"] = daily_sales.index.month
daily_sales["quarter"] = daily_sales.index.quarter
daily_sales["year"] = daily_sales.index.year
daily_sales["dayofmonth"] = daily_sales.index.day

# Step 14: Scale the Sales column
scaler = MinMaxScaler()
daily_sales["Sales_scaled"] = scaler.fit_transform(daily_sales[["Sales"]])

# Add engineered features to improve model performance
daily_sales['lag_1'] = daily_sales['Sales_scaled'].shift(1)
daily_sales['lag_7'] = daily_sales['Sales_scaled'].shift(7)
daily_sales['lag_14'] = daily_sales['Sales_scaled'].shift(14)
daily_sales['rolling_mean_7'] = daily_sales['Sales_scaled'].rolling(window=7).mean()
daily_sales['rolling_std_7'] = daily_sales['Sales_scaled'].rolling(window=7).std()

# Drop rows with NaNs from shifting/rolling
daily_sales.dropna(inplace=True)

# Define input features and target
features = ["Sales_scaled", "lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7"]
X_all = daily_sales[features].values
y_all = daily_sales["Sales_scaled"].values

# Create sequences
window_size = 30
X, y = [], []
for i in range(len(X_all) - window_size):
    X.append(X_all[i:i+window_size])
    y.append(y_all[i+window_size])
X = np.array(X)
y = np.array(y)

# Split into train/test
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
dt_preds = dt_model.predict(X_test.reshape(X_test.shape[0], -1))

# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
rf_preds = rf_model.predict(X_test.reshape(X_test.shape[0], -1))

# XGBoost
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
xgb_preds = xgb_model.predict(X_test.reshape(X_test.shape[0], -1))

def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Results:")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}\n")

# Evaluate models
evaluate_model(y_test, dt_preds, "Decision Tree")
evaluate_model(y_test, rf_preds, "Random Forest")
evaluate_model(y_test, xgb_preds, "XGBoost")

# LSTM Model
lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(units=50, return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(units=1))
lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.fit(X_train, y_train, epochs=10, batch_size=32)
lstm_preds = lstm_model.predict(X_test)
evaluate_model(y_test, lstm_preds, "LSTM")

# GRU Model
gru_model = Sequential()
gru_model.add(GRU(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
gru_model.add(Dropout(0.2))
gru_model.add(GRU(units=50, return_sequences=False))
gru_model.add(Dropout(0.2))
gru_model.add(Dense(units=1))
gru_model.compile(optimizer='adam', loss='mean_squared_error')
gru_model.fit(X_train, y_train, epochs=10, batch_size=32)
gru_preds = gru_model.predict(X_test)
evaluate_model(y_test, gru_preds, "GRU")

# Transformer Model
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout_rate):
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    attention = LayerNormalization(epsilon=1e-6)(attention)
    ff = Dense(ff_dim, activation="relu")(attention)
    ff = Dense(inputs.shape[-1])(ff)
    ff = LayerNormalization(epsilon=1e-6)(ff)
    return ff

input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))
x = transformer_encoder(input_layer, head_size=64, num_heads=4, ff_dim=128, dropout_rate=0.1)
x = GlobalAveragePooling1D()(x)
output_layer = Dense(1)(x)
transformer_model = Model(inputs=input_layer, outputs=output_layer)
transformer_model.compile(optimizer='adam', loss='mean_squared_error')
transformer_model.fit(X_train, y_train, epochs=10, batch_size=32)
transformer_preds = transformer_model.predict(X_test)
evaluate_model(y_test, transformer_preds, "Transformer")

#VISUALIZATIONS

# Plot daily sales trend
plt.figure(figsize=(14, 6))
plt.plot(daily_sales.index, daily_sales["Sales"], color='blue')
plt.title("Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# Boxplot: Sales by Day of Week
plt.figure(figsize=(10, 5))
sns.boxplot(x="dayofweek", y="Sales", data=daily_sales.reset_index())
plt.title("Sales by Day of Week")
plt.xlabel("Day of Week (0=Monday)")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# Boxplot: Sales by Month
plt.figure(figsize=(10, 5))
sns.boxplot(x="month", y="Sales", data=daily_sales.reset_index())
plt.title("Sales by Month")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# Feature importance - Random Forest (aggregated over timesteps)
importances = rf_model.feature_importances_
n_time_steps = X_train.shape[1]  # 30
n_features = X_train.shape[2]    # 6
reshaped_importances = importances.reshape(n_time_steps, n_features)
avg_importances = reshaped_importances.mean(axis=0)

plt.figure(figsize=(8, 4))
sns.barplot(x=avg_importances, y=features)
plt.title("Random Forest - Feature Importance (Averaged Over Time)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Feature importance - XGBoost (aggregated over timesteps)
importances_xgb = xgb_model.feature_importances_
n_time_steps = X_train.shape[1]  # 30
n_features = X_train.shape[2]    # 6
reshaped_xgb_importances = importances_xgb.reshape(n_time_steps, n_features)
avg_xgb_importances = reshaped_xgb_importances.mean(axis=0)

plt.figure(figsize=(8, 4))
sns.barplot(x=avg_xgb_importances, y=features)
plt.title("XGBoost - Feature Importance (Averaged Over Time)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


# Actual vs. Predicted - GRU Model
plt.figure(figsize=(14, 6))
plt.plot(y_test, label="Actual")
plt.plot(gru_preds.flatten(), label="GRU Predicted")
plt.title("GRU Model - Actual vs Predicted Sales")
plt.xlabel("Time Steps")
plt.ylabel("Scaled Sales")
plt.legend()
plt.tight_layout()
plt.show()


Mounted at /content/drive


KeyboardInterrupt: 